# 02 · Preprocessing

Clean the data, normalise text, and create a train/validation/test split ready for modelling.

- **Inputs:** `data/sample/sample_prs.csv`
- **Outputs:** A cleaned table written to `data/interim/` and reusable split logic.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

## 1. Load the raw (sample) data

In [ ]:
from pr_risk.data.load_data import load_csv, save_csv

df = load_csv(SAMPLE_CSV)
print("loaded:", df.shape)
df.head()

## 2. Clean
`preprocess_pr_dataset` wraps `basic_cleaning` (drop duplicates, strip strings, empty → NA).

In [ ]:
from pr_risk.data.preprocess import preprocess_pr_dataset

clean = preprocess_pr_dataset(df)
print("after cleaning:", clean.shape)
clean.head()

## 3. Normalise text
Combine title + description into one normalised `text` field for TF-IDF / embeddings.

In [ ]:
clean["text"] = (
    clean["title"].fillna("").astype(str) + " " + clean["description"].fillna("").astype(str)
).str.lower().str.strip()
clean[["title", "description", "text"]].head()

## 4. Train / validation / test split
The package provides a **stratified** `create_train_val_test_split` for the real
dataset. On the tiny sample, stratification fails (singleton classes), so we catch
that and fall back to a simple split — this is exactly the kind of `TODO` boundary
between sample and real data.

In [ ]:
from sklearn.model_selection import train_test_split

from pr_risk.data.split_data import create_train_val_test_split

# Binary target {0, 1} is the runnable modelling target on the sample.
binary = clean[clean["is_risky"].isin([0, 1])].copy()

try:
    train_df, val_df, test_df = create_train_val_test_split(binary, target_col="is_risky")
    print("Stratified split OK (this is the path used on the real dataset).")
except ValueError as exc:
    print("Stratified split not possible on the tiny sample:\n  ", exc)
    print("Falling back to a simple non-stratified split for the demo.")
    train_df, test_df = train_test_split(binary, test_size=0.3, random_state=42)
    val_df = test_df.copy()

print("sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

## 5. Persist the cleaned table to `data/interim/`
`data/interim/` is gitignored — these are regenerable artifacts.

In [ ]:
interim_path = PROJECT_ROOT / "data" / "interim" / "sample_clean.csv"
save_csv(clean, interim_path)
print("wrote:", interim_path.relative_to(PROJECT_ROOT))

## Next steps / TODO (real data)
- Implement PRismBench-specific normalisation in `pr_risk.data.preprocess` / `clean_data` (the current versions carry `TODO`s).
- Handle missing values and dtype coercion for the real columns.
- Use the **stratified** `create_train_val_test_split` once classes have enough members.
- Continue to **03 · Baseline Models**.